In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import pandas as pd
from src.utils.pipeline import load_all_snapshots
from src.data.validation import plate_appearances
from src.features.plate_discipline import add_discipline_flags
from src.models.marcel import season_lines, project, evaluate

SEASONS = [2021, 2022, 2023, 2024]
df = load_all_snapshots(seasons=SEASONS)
df["season"] = pd.to_datetime(df["game_date"]).dt.year

pa = plate_appearances(df)
pitcher_ids = set(df["pitcher"].dropna().unique())
lines = season_lines(pa, pitcher_ids=pitcher_ids)

# Skill rates per batter-season
f = add_discipline_flags(df)
f = f[~f["batter"].isin(pitcher_ids)]

sw = f[f["is_swing"]]
oz = f[~f["in_zone"]]
zs = f[f["is_swing"] & f["in_zone"]]

skills = pd.DataFrame({
    "chase_pct": oz.groupby(["batter", "season"])["is_swing"].mean(),
    "zone_swing_pct": f[f["in_zone"]].groupby(["batter", "season"])["is_swing"].mean(),
    "whiff_pct": sw.groupby(["batter", "season"])["is_whiff"].mean(),
    "zone_contact_pct": 1 - zs.groupby(["batter", "season"])["is_whiff"].mean(),
    "swing_pct": f.groupby(["batter", "season"])["is_swing"].mean(),
    "n_oz": oz.groupby(["batter", "season"]).size(),
    "n_zone_swings": zs.groupby(["batter", "season"]).size(),
}).reset_index()

print(skills.groupby("season").size().to_dict())
print(skills[["chase_pct", "zone_contact_pct", "whiff_pct"]].describe().round(3).to_string())

{2021: 513, 2022: 542, 2023: 523, 2024: 548}
       chase_pct  zone_contact_pct  whiff_pct
count   2124.000          2124.000   2126.000
mean       0.289             0.829      0.253
std        0.082             0.083      0.087
min        0.000             0.000      0.000
25%        0.238             0.794      0.200
50%        0.283             0.838      0.249
75%        0.330             0.877      0.296
max        1.000             1.000      1.000


In [2]:
TARGET_SEASON = 2024
PRIOR = [2021, 2022, 2023]

# Marcel projection for the target season — this becomes a FEATURE.
marcel_k = project(lines, TARGET_SEASON, "k")
marcel_bb = project(lines, TARGET_SEASON, "bb")

# Skills from the season immediately before the target only.
# Using target-season skills would be leakage: they are not knowable
# when the projection is made.
skills_prior = skills[skills["season"] == TARGET_SEASON - 1].set_index("batter")

actual = lines[lines["season"] == TARGET_SEASON].set_index("batter")
actual["k_pct"] = actual["k"] / actual["pa_count"]
actual["bb_pct"] = actual["bb"] / actual["pa_count"]

data = (
    marcel_k[["projection", "weighted_pa", "league_rate"]]
    .rename(columns={"projection": "marcel_k"})
    .join(marcel_bb["projection"].rename("marcel_bb"))
    .join(skills_prior.drop(columns=["season"]))
    .join(actual[["k_pct", "bb_pct", "pa_count"]])
)

data = data[(data["pa_count"] >= 200) & (data["weighted_pa"] >= 100)
            & data["chase_pct"].notna()]
print(f"{len(data)} batters with Marcel, prior skills, and 2024 outcome")

254 batters with Marcel, prior skills, and 2024 outcome


In [3]:
from sklearn.linear_model import LinearRegression

SKILL_FEATURES = ["chase_pct", "zone_swing_pct", "whiff_pct",
                  "zone_contact_pct", "swing_pct"]

def fit_and_eval(target, marcel_col, extra_features):
    X_base = data[[marcel_col]]
    X_full = data[[marcel_col] + extra_features]
    y = data[target]

    base = LinearRegression().fit(X_base, y)
    full = LinearRegression().fit(X_full, y)

    return pd.DataFrame([
        {"model": "Marcel (raw)", **evaluate(data[marcel_col], y)},
        {"model": "Marcel rescaled", **evaluate(pd.Series(base.predict(X_base), index=data.index), y)},
        {"model": "Marcel + skills", **evaluate(pd.Series(full.predict(X_full), index=data.index), y)},
    ]), full

res_k, model_k = fit_and_eval("k_pct", "marcel_k", SKILL_FEATURES)
print("=== K% ===")
print(res_k.round(4).to_string(index=False))
print()
print(pd.Series(model_k.coef_, index=["marcel_k"] + SKILL_FEATURES).round(4).to_string())

=== K% ===
          model    mae   rmse   corr   n
   Marcel (raw) 0.0271 0.0348 0.8066 254
Marcel rescaled 0.0268 0.0346 0.8066 254
Marcel + skills 0.0261 0.0332 0.8233 254

marcel_k            0.7999
chase_pct          -0.2039
zone_swing_pct     -0.0672
whiff_pct           0.1442
zone_contact_pct   -0.1480
swing_pct           0.1893


In [4]:
from sklearn.model_selection import KFold, cross_val_predict

def cv_eval(target, marcel_col, extra_features, n_splits=5):
    y = data[target]
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    rows = [{"model": "Marcel (raw)", **evaluate(data[marcel_col], y)}]
    for name, cols in [("Marcel rescaled", [marcel_col]),
                       ("Marcel + skills", [marcel_col] + extra_features)]:
        pred = cross_val_predict(LinearRegression(), data[cols], y, cv=kf)
        rows.append({"model": name, **evaluate(pd.Series(pred, index=data.index), y)})
    return pd.DataFrame(rows)

print("=== K% (5-fold CV) ===")
print(cv_eval("k_pct", "marcel_k", SKILL_FEATURES).round(4).to_string(index=False))
print()
print("=== BB% (5-fold CV) ===")
print(cv_eval("bb_pct", "marcel_bb", SKILL_FEATURES).round(4).to_string(index=False))

=== K% (5-fold CV) ===
          model    mae   rmse   corr   n
   Marcel (raw) 0.0271 0.0348 0.8066 254
Marcel rescaled 0.0269 0.0349 0.8027 254
Marcel + skills 0.0269 0.0345 0.8083 254

=== BB% (5-fold CV) ===
          model    mae   rmse   corr   n
   Marcel (raw) 0.0144 0.0180 0.7596 254
Marcel rescaled 0.0142 0.0176 0.7564 254
Marcel + skills 0.0139 0.0173 0.7637 254


In [5]:
print(data[SKILL_FEATURES].corr().round(2).to_string())
print()

from numpy.linalg import cond
X = data[SKILL_FEATURES].apply(lambda s: (s - s.mean()) / s.std())
print("condition number:", round(cond(X.to_numpy()), 1))

                  chase_pct  zone_swing_pct  whiff_pct  zone_contact_pct  swing_pct
chase_pct              1.00            0.55       0.29             -0.11       0.88
zone_swing_pct         0.55            1.00       0.30             -0.22       0.85
whiff_pct              0.29            0.30       1.00             -0.91       0.30
zone_contact_pct      -0.11           -0.22      -0.91              1.00      -0.17
swing_pct              0.88            0.85       0.30             -0.17       1.00

condition number: 14.2


In [6]:
MINIMAL = ["chase_pct", "zone_contact_pct"]

for target, mcol in [("k_pct", "marcel_k"), ("bb_pct", "marcel_bb")]:
    print(f"=== {target} ===")
    print(cv_eval(target, mcol, MINIMAL).round(4).to_string(index=False))
    print()

=== k_pct ===
          model    mae   rmse   corr   n
   Marcel (raw) 0.0271 0.0348 0.8066 254
Marcel rescaled 0.0269 0.0349 0.8027 254
Marcel + skills 0.0265 0.0340 0.8142 254

=== bb_pct ===
          model    mae   rmse   corr   n
   Marcel (raw) 0.0144 0.0180 0.7596 254
Marcel rescaled 0.0142 0.0176 0.7564 254
Marcel + skills 0.0137 0.0172 0.7669 254



In [7]:
from sklearn.linear_model import LinearRegression

for target, mcol in [("k_pct", "marcel_k"), ("bb_pct", "marcel_bb")]:
    cols = [mcol] + MINIMAL
    m = LinearRegression().fit(data[cols], data[target])
    print(f"{target}:")
    print(pd.Series(m.coef_, index=cols).round(4).to_string())
    print()

k_pct:
marcel_k            0.8577
chase_pct          -0.0795
zone_contact_pct   -0.2670

bb_pct:
marcel_bb           0.7368
chase_pct          -0.0977
zone_contact_pct   -0.0526



In [8]:
cols_k = ["marcel_k"] + MINIMAL
pred_skill = pd.Series(
    cross_val_predict(LinearRegression(), data[cols_k], data["k_pct"],
                      cv=KFold(5, shuffle=True, random_state=42)),
    index=data.index)

diff = (data["marcel_k"] - data["k_pct"]).abs() - (pred_skill - data["k_pct"]).abs()
data["improvement"] = diff

# Where does the skill model help most?
print("improvement by Marcel error size:")
data["marcel_err"] = (data["marcel_k"] - data["k_pct"]).abs()
buckets = pd.qcut(data["marcel_err"], 4)
print(data.groupby(buckets, observed=True)["improvement"].agg(["mean", "size"]).round(5).to_string())

improvement by Marcel error size:
                         mean  size
marcel_err                         
(-0.0009723, 0.0104] -0.00420    64
(0.0104, 0.0215]     -0.00026    63
(0.0215, 0.0386]      0.00252    63
(0.0386, 0.12]        0.00449    64
